# Import and setup

In [2]:
from __future__ import annotations

import argparse
import os
import sys
import uuid
from pathlib import Path

from dotenv import load_dotenv


In [3]:
current_file_path = Path().resolve()
sys.path.insert(0, str(current_file_path.parent))

# Add the project's `src` directory to the Python path
_REPO_ROOT = current_file_path.parent


_SRC_DIR = _REPO_ROOT / "src"
if str(_SRC_DIR) not in sys.path:
    sys.path.insert(0, str(_SRC_DIR))


In [4]:

from support_agent.config import Settings
from support_agent.cortex_agent.service import chat_with_conversation
from support_agent.snowflake_client import create_snowpark_session
from support_agent.config import get_settings


# Main code

## Sesson loading

In [5]:
# Load environment variables from .env file
load_dotenv()

# Load application settings
settings = get_settings()


# Create a Snowpark session
print("Connecting to Snowflake...")
session = create_snowpark_session(settings)
print("Connection successful.")


Connecting to Snowflake...
Connection successful.


## Agent workflow

In [6]:
query = "Please provide information on techniques and tools for digital brand growth strategies, specifically focused on increasing brand awareness and engagement."

### Main workflow interaction with agent instanciation 

In [13]:
# This single function call encapsulates the entire workflow:
# - It uses ConversationManager to get or create a Cortex Thread.
# - It calls the Cortex Agent via the REST client within that thread.
# - It persists the updated thread state for the next turn.
print("Sending query to Cortex Agent...")
result = chat_with_conversation(
    session=session,
    settings=settings,
    conversation_id=None,
    user_text=query,
    stream=False,  # Use False for a simple script to get the full response
)
print("Agent has responded.")


Sending query to Cortex Agent...
=> Created new thread 34744956 for conversation None
Agent has responded.


In [7]:
from support_agent.cortex_agent.conversation_manager import ConversationManager

conversation_id = None
user_text = query
stream: bool = True


In [9]:
from support_agent.cortex_agent.service import get_cortex_rest_client

In [10]:

"""Send a user message using conversation-based thread management.

This function automatically:
1. Retrieves or creates a thread for the conversation
2. Tracks parent_message_id from database
3. Updates conversation state after the response

Args:
    session: Snowpark session for database access
    settings: Application settings
    conversation_id: Unique conversation identifier (e.g., session ID, user ID)
    user_text: The user message content
    stream: If true, uses SSE streaming to capture message IDs

Returns:
    AgentRunResult containing assistant text and message IDs

Example:
    >>> result = chat_with_conversation(
    ...     session=session,
    ...     settings=settings,
    ...     conversation_id="session_abc123",
    ...     user_text="How do I reset my password?"
    ... )
    >>> print(result.assistant_text)
"""
manager = ConversationManager(session, settings)
rest_client = get_cortex_rest_client(settings)


In [11]:

# Get or create thread with proper parent_message_id
thread_id, parent_message_id = manager.get_or_create_thread(
    conversation_id="test_id_notebook",
    rest_client=rest_client,
)


=> Reusing thread 34744964 for conversation test_id_notebook


In [12]:
thread_db = session.sql(f"""
SELECT * FROM {settings.database}.APP.CORTEX_CONVERSATIONS
""").collect()

In [13]:
thread_db

[Row(CONVERSATION_ID='c368468d', THREAD_ID=34744968, LAST_ASSISTANT_MESSAGE_ID=8894679049, ORIGIN_APPLICATION='tickets_app', CREATED_AT=datetime.datetime(2026, 2, 28, 12, 16, 24, 698000), UPDATED_AT=datetime.datetime(2026, 2, 28, 12, 18, 26, 359000)),
 Row(CONVERSATION_ID='test_id_notebook', THREAD_ID=34744964, LAST_ASSISTANT_MESSAGE_ID=None, ORIGIN_APPLICATION='tickets_app', CREATED_AT=datetime.datetime(2026, 2, 22, 12, 4, 36, 301000), UPDATED_AT=datetime.datetime(2026, 2, 22, 12, 4, 36, 301000)),
 Row(CONVERSATION_ID='b476d37d', THREAD_ID=34744948, LAST_ASSISTANT_MESSAGE_ID=8894678281, ORIGIN_APPLICATION='tickets_app', CREATED_AT=datetime.datetime(2026, 2, 21, 10, 27, 36, 804000), UPDATED_AT=datetime.datetime(2026, 2, 21, 10, 29, 4, 204000)),
 Row(CONVERSATION_ID='ef712a8b', THREAD_ID=34744944, LAST_ASSISTANT_MESSAGE_ID=8894678265, ORIGIN_APPLICATION='tickets_app', CREATED_AT=datetime.datetime(2026, 2, 21, 10, 25, 5, 331000), UPDATED_AT=datetime.datetime(2026, 2, 21, 10, 26, 2, 99800

In [14]:
user_text

'Please provide information on techniques and tools for digital brand growth strategies, specifically focused on increasing brand awareness and engagement.'

In [ ]:
# Send message
result = rest_client.run_agent_with_object(
    database=settings.database,
    schema=settings.cortex_agent_schema,
    agent_name=settings.cortex_agent_name,
    thread_id=thread_id,
    parent_message_id=parent_message_id,
    user_text=user_text,
    stream=stream
)


In [17]:
result.raw_response

{'content': [{'thinking': {'text': 'I will use cortex search to find relevant documents to this question.'},
   'type': 'thinking'},
  {'tool_use': {'client_side_execute': False,
    'input': {'min_confidence_score': 0.5,
     'query': 'Please provide information on techniques and tools for digital brand growth strategies, specifically focused on increasing brand awareness and engagement.'},
    'name': 'cortex_search',
    'tool_use_id': 'tooluse_2f6d47e80d0049e7a16eaf',
    'type': 'cortex_search'},
   'type': 'tool_use'},
  {'tool_result': {'content': [{'json': {'message': 'No results found.',
       'searchResultsPersist': {}},
      'type': 'json'}],
    'name': 'cortex_search',
    'status': 'success',
    'tool_use_id': 'tooluse_2f6d47e80d0049e7a16eaf',
    'type': 'cortex_search'},
   'type': 'tool_result'},
  {'thinking': {'text': "\nThe Cortex Search didn't return any results for the specific query about digital brand growth strategies. Since no relevant context was found in 

In [ ]:

# Update conversation state if we got an assistant message ID
if result.assistant_message_id is not None:
    manager.update_conversation(
        conversation_id=conversation_id,
        thread_id=thread_id,
        assistant_message_id=result.assistant_message_id,
    )

return result


In [17]:
from support_agent.cortex_agent.rest_client import (
    AgentRunResult,
    CortexAgentsRestClient,
)


def get_cortex_rest_client(settings: Settings) -> CortexAgentsRestClient:
    """Create a CortexAgentsRestClient from Settings.

    Raises:
        ValueError: If required REST settings are not configured.
    """
    return CortexAgentsRestClient(
        account_url=settings.snowflake_account_url or settings.account,
        token=settings.snowflake_rest_token,
        origin_application=settings.cortex_origin_application,
    )



### Fallaback without threading

In [ ]:
from support_agent.cortex_agent.rest_client import CortexAgentsRestClient
# Fallback: stateless mode with full message history
client = CortexAgentsRestClient(
    account_url=settings.snowflake_account_url,
    token=settings.snowflake_rest_token,
    origin_application=settings.cortex_origin_application,
)
messages = [
    {
        "role": m["role"],
        "content": [{"type": "text", "text": m["content"]}],
    }
    for m in st.session_state.chat_messages
]
result = client.run_agent_with_object_messages(
    database=settings.database,
    schema=settings.cortex_agent_schema,
    agent_name=settings.cortex_agent_name,
    messages=messages,
    stream=False,
)

In [15]:

# Print the results
print("--- Agent Response ---")
print(result.assistant_text.strip())
print("----------------------")

--- Agent Response ---
I can help you with your VPN problem. Based on common VPN issues, here are some troubleshooting steps:

**Quick Fixes:**
1. **Restart your VPN connection** - Disconnect and reconnect
2. **Reboot your device** - Turn it off and back on
3. **Check your internet connection** - Ensure you have stable internet before connecting to VPN
4. **Update VPN software** - Make sure you're running the latest version
5. **Try a different VPN server** - Switch to another server location if available

**Common Causes:**
- Incorrect VPN configuration settings
- Recent software updates or network configuration changes
- VPN-router misconfiguration

**What specific issue are you experiencing?**
- VPN won't connect at all?
- Connected but no internet access?
- Slow/unstable connection?
- Error messages appearing?

Let me know the details so I can provide more targeted help.
----------------------


# Close session

In [ ]:

session.close()
print("Snowflake connection closed.")